# 03 — Setup and Prerequisites

> **DataMindAI | Ahmed**

## Account requirements

| Account type | Supports Jobs? | Recommendation |
|-------------|---------------|----------------|
| Community Edition | ❌ No | Do not use for this series |
| Free Edition | ✅ Yes | **Use this** — sign up at databricks.com/try |
| Standard / Premium | ✅ Yes | For enterprise use |

## Required settings in Databricks UI
1. **Settings → Developer → Enable experimental features** ← must enable this
2. This unlocks: If/Else tasks, For-Each tasks, Lakeflow Designer

## What you need to know
- ✅ Basic Python (print, variables, lists, dicts)
- ✅ Basic SQL (SELECT, GROUP BY, WHERE)
- ✅ What ETL means
- ❌ PySpark NOT required for orchestration logic

---
## Environment Check
---

In [ ]:
# ── Check your Spark and Databricks version ──────────────────────────────────
print('=== ENVIRONMENT CHECK ===')
print(f'Spark version    : {spark.version}')
print(f'Python version   : {__import__("sys").version.split()[0]}')
print()

# Check Unity Catalog is available
try:
    catalogs = [r[0] for r in spark.sql('SHOW CATALOGS').collect()]
    print(f'Available catalogs: {catalogs}')
    print('Unity Catalog    : ENABLED')
except Exception as e:
    print(f'Unity Catalog    : NOT AVAILABLE — {e}')
    print('  You may need to enable Unity Catalog in your workspace settings.')
print('========================')

In [ ]:
# ── Create the full catalog and schema structure for this series ──────────────
catalog = 'demo_catalog'  # change this to your catalog name

schemas = ['bronze', 'silver', 'gold', 'config']

spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalog}')
for schema in schemas:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}')
    print(f'  Created: {catalog}.{schema}')

print()
print(f'All schemas ready under catalog: {catalog}')

---
## Seed Data — Used Across All 10 Notebooks
---

In [ ]:
# ── Master student dataset ────────────────────────────────────────────────────
# This exact dataset is used as the demo data in every notebook.
# Run this cell once to make it available throughout the series.

from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp

students = [
    Row(student_id='S001', name='Aisha Rahman',    course='Data Science', score=82, attendance=95, fee_paid=True,  year=2),
    Row(student_id='S002', name='James Bradley',   course='Data Science', score=41, attendance=60, fee_paid=True,  year=1),
    Row(student_id='S003', name='Priya Patel',     course='Engineering',  score=76, attendance=88, fee_paid=True,  year=3),
    Row(student_id='S004', name='Carlos Mendez',   course='Engineering',  score=55, attendance=72, fee_paid=False, year=2),
    Row(student_id='S005', name='Sophie Williams', course='Mathematics',  score=91, attendance=98, fee_paid=True,  year=1),
    Row(student_id='S006', name='Kwame Asante',    course='Mathematics',  score=38, attendance=55, fee_paid=True,  year=2),
    Row(student_id='S007', name='Liu Wei',         course='Data Science', score=67, attendance=80, fee_paid=False, year=3),
    Row(student_id='S008', name='Emma Johnson',    course='Engineering',  score=74, attendance=85, fee_paid=True,  year=1),
    Row(student_id='S009', name='Tariq Ahmed',     course='Data Science', score=29, attendance=40, fee_paid=True,  year=2),
    Row(student_id='S010', name='Fatima Al-Sayed', course='Mathematics',  score=88, attendance=94, fee_paid=True,  year=3),
]

df = (spark.createDataFrame(students)
        .withColumn('loaded_at', current_timestamp()))

df.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.bronze.students_raw')
print(f'Master dataset written: {df.count()} students')
df.show()

In [ ]:
# ── Course mapping table (used in For-Each and SQL demos) ─────────────────────
from pyspark.sql import Row

courses = [
    Row(course_id='C001', course_name='Data Science', department='Computing',  credits=120, pass_mark=40),
    Row(course_id='C002', course_name='Engineering',  department='Technology', credits=150, pass_mark=40),
    Row(course_id='C003', course_name='Mathematics',  department='Science',    credits=120, pass_mark=40),
]

df_courses = spark.createDataFrame(courses)
df_courses.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.config.courses')
print('Course mapping table written:')
df_courses.show()

In [ ]:
# ── Assessment mapping table (used in dynamic ingestion demo) ─────────────────
from pyspark.sql import Row

files = [
    Row(file_id='F001', file_name='ds_assessment_jan.csv',  course='Data Science', period='Jan-2024'),
    Row(file_id='F002', file_name='eng_assessment_jan.csv', course='Engineering',  period='Jan-2024'),
    Row(file_id='F003', file_name='maths_assessment_jan.csv',course='Mathematics', period='Jan-2024'),
]

df_files = spark.createDataFrame(files)
df_files.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.config.file_mapping')
print('File mapping table written:')
df_files.show()

print()
print('All seed data ready. Proceed to 04_Basic_Tasks_and_DAGs.ipynb')